In [ ]:
import torch
from tqdm import tqdm
import re
torch.set_default_device("cuda")

# 1. Parse Text Data Cleanly
data = open("input.txt", "r").read()
token_pattern = r"\w+| |\n|[.,!?;:]"
list_words = re.findall(token_pattern, data)
words = list(set(list_words))
data_size, vocab_size = len(list_words), len(words)
words_to_ix = {c: i for i, c in enumerate(words)}
ix_to_words = {i: c for i, c in enumerate(words)}

# Hyperparameters
hidden_size = 100
seq_size = 7
learning_rate = 0.001

# Model parameters 
Whx = torch.randn(hidden_size, vocab_size) * 0.01
Whh = torch.randn(hidden_size, hidden_size) * 0.01
Why = torch.randn(vocab_size, hidden_size) * 0.01
bh = torch.zeros((hidden_size, 1))
by = torch.zeros((vocab_size, 1))

def lossFunc(inputs, targets, hprev):
    xs, ys, hs, ps = {}, {}, {}, {}
    hs[-1] = torch.clone(hprev)
    loss = 0.0

    # Forward Pass
    for t in range(len(inputs)):
        xs[t] = torch.zeros((vocab_size, 1))
        xs[t][inputs[t]] = 1

       
        hs[t] = torch.tanh(Whx @ xs[t] + Whh @ hs[t - 1] + bh)
        ys[t] = Why @ hs[t] + by

        exp_y = torch.exp(ys[t] - torch.max(ys[t]))
        ps[t] = exp_y / torch.sum(exp_y)

        prob = torch.clamp(ps[t][targets[t], 0], min=1e-12)
        loss += -torch.log(prob).item()

    # Backward Pass 
    dWhx = torch.zeros_like(Whx)
    dWhh = torch.zeros_like(Whh)
    dWhy = torch.zeros_like(Why)
    dbh = torch.zeros_like(bh)
    dby = torch.zeros_like(by)
    dhnext = torch.zeros_like(hs[0])

    for t in reversed(range(len(inputs))):
        dy = torch.clone(ps[t])
        dy[targets[t]] -= 1 

        dWhy += dy @ hs[t].T
        dby += dy

        dh = Why.T @ dy + dhnext
        dhraw = (1 - hs[t] ** 2) * dh  

        dbh += dhraw
        dWhx += dhraw @ xs[t].T        
        dWhh += dhraw @ hs[t - 1].T

        dhnext = Whh.T @ dhraw

    # Normalize gradients by sequence length to ensure stable updates
    for dparam in [dWhx, dWhh, dWhy, dbh, dby]:
        dparam /= len(inputs)
        dparam.clamp_(-5, 5)

    return (loss / len(inputs)), dWhx, dWhh, dWhy, dbh, dby, hs[len(inputs) - 1]

n, p = 0, 0
smooth_loss = -torch.log(torch.tensor(1.0 / vocab_size)).item()

# Initialize Scratch Memory Tensors for Adam
mWhx = torch.zeros_like(Whx)
mWhh = torch.zeros_like(Whh)
mWhy = torch.zeros_like(Why)
mbh = torch.zeros_like(bh)
mby = torch.zeros_like(by)

vWhx = torch.zeros_like(Whx)
vWhh = torch.zeros_like(Whh)
vWhy = torch.zeros_like(Why)
vbh = torch.zeros_like(bh)
vby = torch.zeros_like(by)

p_bar = tqdm(range(50000), desc="epoch = 0", colour="red")
beta1 = 0.9
beta2 = 0.999
t = 0

for epoch_idx in p_bar:
    # Fixed Data Splitting Boundary Condition
    if p + seq_size + 1 >= len(list_words) or n == 0:
        hprev = torch.zeros((hidden_size, 1))
        p = 0

    inputs = [words_to_ix[ch] for ch in list_words[p : p + seq_size]]
    targets = [words_to_ix[ch] for ch in list_words[p + 1 : p + seq_size + 1]]

    p_bar.set_description(f"Step {epoch_idx}")

    loss, dWhx, dWhh, dWhy, dbh, dby, hprev = lossFunc(inputs, targets, hprev)
    smooth_loss = smooth_loss * 0.999 + loss * 0.001
    p_bar.set_postfix(loss=f'{smooth_loss:.4f}')
    t += 1
    
    # 100% Manual Adam Updates In-Place
    with torch.no_grad():
        for param, dparam, m_mems, v_mems in zip(
            [Whx, Whh, Why, bh, by],
            [dWhx, dWhh, dWhy, dbh, dby],
            [mWhx, mWhh, mWhy, mbh, mby],
            [vWhx, vWhh, vWhy, vbh, vby],
        ):
            m_mems[:] = beta1 * m_mems + (1 - beta1) * dparam
            v_mems[:] = beta2 * v_mems + (1 - beta2) * dparam**2

            m_corrected = m_mems / (1 - beta1**t)
            v_corrected = v_mems / (1 - beta2**t)

            param += -learning_rate * m_corrected / (torch.sqrt(v_corrected) + 1e-8)

    p += seq_size
    n += 1


Step 49999: 100%|██████████| 50000/50000 [42:49<00:00, 19.46it/s, loss=1.9816]


In [25]:
torch.save({"Whx":Whx,"Whh":Whh,"Why":Why,"bh":bh,"by":by,},open("model_weights.pkl",'wb'))

In [26]:
def sample_scratch(h, seed_word, n):
    """Generates and stitches raw word-tokens from scratch tensors"""
    current_idx = words_to_ix.get(seed_word, 0)
    x = torch.zeros((vocab_size, 1))
    x[current_idx] = 1
    
    generated_tokens = [seed_word]
    
    with torch.no_grad():
        for t in range(n):
            h = torch.tanh(Whx @ x + Whh @ h + bh)
            y = Why @ h + by
            exp_y = torch.exp(y - torch.max(y))
            p = exp_y / torch.sum(exp_y)
            p = p.flatten()
            ix = torch.multinomial(p, 1).item()
            
            x = torch.zeros((vocab_size, 1))
            x[ix] = 1
            generated_tokens.append(ix_to_words[ix])
            
    # Format tokens cleanly back to a string layout
    text = ""
    for token in generated_tokens:
        if token in [".", ",", "!", "?", ";", ":"]:
            text += token
        elif token in [" ", "\n"]:
            text += token
        else:
            if text and text[-1] not in [" ", "\n"]:
                text += " " + token
            else:
                text += token
    return text

# Test it out in your secondary cell
initial_hidden = torch.zeros((hidden_size, 1))
print(sample_scratch(initial_hidden, "KING", 500))


KING, to find him from his own stomach,.
    I am my hand.
  HELENA. When Enter BERTRAM,
    Attend to my estate hopes of stand
    When most credit, and hath old have itself d, when help call.
  hearts Quicken most remedy in the time theft is you,
    To virtuous particular old start of my credit to not
                 Enter LAFEU. young dare our house,
    And approv:
    With commit
    hearing receipt
    And about to sword
    A Florentines one the medicine,
    But But mine own themselves The aged broke had impress else,.          Exit      I When this Majesty.
    But grapes, and brave art for me more my fellow, and well call by hence,
    To ability this gain, sword
    The tender the repent.
                                          Exit PAROLLES
    Do and more Smile any curiously? ll pray him,
                 Exeunt
Florence ACT was help; the other hath Rousillon,
    No, one, haste and my father put me three that we her him;
    But give all pen
    And should all my
